# PHOENIX14T–CSLR: 68 lượt chạy — Kaggle

Run All chuẩn bị dữ liệu chung rồi chạy 67 cấu hình định lượng và một baseline cho so sánh gloss định tính. Mỗi thí nghiệm có checkpoint riêng trên Drive. Chạy lại tiếp tục thí nghiệm dở, bỏ qua thí nghiệm hoàn tất. Đây là tái dựng theo paper, có giả định công khai trong `docs/PHOENIX14T_SUITE.md`; chưa chứng minh đạt số paper.

## Chuẩn bị một lần
Đặt `SignLanguage-Reproduce-portable.zip` vào thư mục Drive chung. Tạo OAuth Kaggle Secret một lần theo `notebooks/DRIVE_SETUP.md`; sau đó notebook tự tìm ZIP mã và tự tải/cache dữ liệu PHOENIX14T.

## Cấu hình

Đã điền ID thư mục Drive bạn cung cấp. Mặc định chạy đủ 68 lượt PHOENIX14T–CSLR qua nhiều phiên có checkpoint.

In [ ]:
from pathlib import Path
import time
WORKSPACE=Path("/kaggle/working")
DRIVE_WORKSPACE_FOLDER_ID="1RpGbT-9SXIr21OVfVmRRn4K_XhRulCeL"
DATA_ARCHIVE_NAME="phoenix-2014-T.v3.tar.gz"
DATA_ROOT=Path("/tmp/phoenix14t")
MANIFEST_ROOT=WORKSPACE/"manifests/phoenix14t"
PROJECT_ROOT=WORKSPACE/"SignLanguage-Reproduce"
project_root=PROJECT_ROOT

DATASET="phoenix14t"
PHASE="cslr"
WORKERS=0  # read one batch at a time to limit host RAM
EPOCHS=50
SESSION_MINUTES=600  # entire notebook session, including preparation
ONLY_EXPERIMENTS=""  # empty: all 68; e.g. "main,modules_none"
CAMPAIGN_NAME="phoenix14t_cslr_suite_checkpoint_v12_seed0"
NOTEBOOK_START=time.monotonic()

## Kiểm tra và cài thư viện

Dữ liệu Input chỉ đọc; checkpoint ghi trong `/kaggle/working` và được giữ trong Output khi tạo Version. Không đặt checkpoint vào `/kaggle/input`.

In [ ]:
%pip install -q google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import json, os, sys, zipfile, subprocess
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

try:
    oauth_json=UserSecretsClient().get_secret("GOOGLE_DRIVE_OAUTH_JSON")
except Exception as exc:
    raise RuntimeError("Add your private Kaggle Secret GOOGLE_DRIVE_OAUTH_JSON first; see notebooks/DRIVE_SETUP.md") from exc

credentials=Credentials.from_authorized_user_info(json.loads(oauth_json),scopes=["https://www.googleapis.com/auth/drive"])
if not credentials.valid: credentials.refresh(Request())
drive_service=build("drive","v3",credentials=credentials,cache_discovery=False)
del oauth_json,credentials
def drive_download(file_id,destination):
    destination=Path(destination); destination.parent.mkdir(parents=True,exist_ok=True)
    temp=destination.with_name(destination.name+".download-tmp")
    with temp.open("wb") as stream:
        downloader=MediaIoBaseDownload(stream,drive_service.files().get_media(fileId=file_id),chunksize=16*1024*1024)
        done=False
        while not done: _,done=downloader.next_chunk()
    os.replace(temp,destination)
WORKSPACE.mkdir(parents=True,exist_ok=True)
project_matches=drive_service.files().list(q=f"'{DRIVE_WORKSPACE_FOLDER_ID}' in parents and name='SignLanguage-Reproduce-portable.zip' and trashed=false",fields="files(id,name,mimeType)",pageSize=10,supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get("files",[])
if len(project_matches)!=1: raise FileNotFoundError("Put exactly one SignLanguage-Reproduce-portable.zip in the shared Drive folder")
project_drive_file=project_matches[0]
DRIVE_PROJECT_FILE_ID=project_drive_file["id"]
if project_drive_file["mimeType"]!="application/zip" and not project_drive_file["name"].lower().endswith(".zip"):
    raise ValueError("The Drive project file must be the portable ZIP")
PROJECT_ZIP=WORKSPACE/project_drive_file["name"]
drive_download(DRIVE_PROJECT_FILE_ID,PROJECT_ZIP)
with zipfile.ZipFile(PROJECT_ZIP) as archive:
    for entry in archive.infolist():
        if not (WORKSPACE/entry.filename).resolve().is_relative_to(WORKSPACE.resolve()): raise ValueError("Unsafe project ZIP path")
    archive.extractall(WORKSPACE)
PROJECT_ROOT=WORKSPACE/"SignLanguage-Reproduce"
if not (PROJECT_ROOT/"scripts/launch.py").is_file(): raise FileNotFoundError("Unexpected project ZIP layout")
sys.path.insert(0,str(PROJECT_ROOT))
subprocess.run([sys.executable,"scripts/verify_bundle.py"],cwd=PROJECT_ROOT,check=True)
from scripts.kaggle_drive import download_file,ensure_folder,find_child,upsert_file
print("Project ZIP downloaded from the shared Drive folder.")

In [ ]:
import zipfile, shutil, subprocess, sys
sys.path.insert(0,str(PROJECT_ROOT))
from scripts.kaggle_drive import download_file,ensure_folder,find_child,upsert_file
subprocess.run([sys.executable,"scripts/verify_bundle.py"],cwd=PROJECT_ROOT,check=True)
DRIVE_RUNS_FOLDER_ID=ensure_folder(drive_service,DRIVE_WORKSPACE_FOLDER_ID,"runs")
archive_matches=find_child(drive_service,DRIVE_WORKSPACE_FOLDER_ID,DATA_ARCHIVE_NAME)
archive_path=Path("/tmp")/DATA_ARCHIVE_NAME
if archive_path.is_file():
    print("Reusing the archive already downloaded in this Kaggle session.")
elif archive_matches:
    if len(archive_matches)>1: raise ValueError("Duplicate PHOENIX archives in shared Drive folder")
    print("Loading the cached PHOENIX archive from Drive.")
    download_file(drive_service,archive_matches[0]["id"],archive_path)
else:
    from scripts.fetch_phoenix import download_resumable,URL
    print("The Drive cache is empty; downloading the official PHOENIX archive.")
    download_resumable(URL,archive_path)
    quota=drive_service.about().get(fields="storageQuota").execute().get("storageQuota",{})
    limit=int(quota.get("limit") or 0); usage=int(quota.get("usage") or 0)
    available=limit-usage if limit else None
    if available is None or available>=archive_path.stat().st_size:
        try:
            upsert_file(drive_service,DRIVE_WORKSPACE_FOLDER_ID,archive_path,DATA_ARCHIVE_NAME)
            print("Cached the archive in the shared Drive folder for later sessions.")
        except Exception as exc:
            print(f"Could not cache archive to Drive ({exc}); continuing this session with the local copy.")
    else:
        print("Drive has less free space than the archive; continuing locally. Kaggle may need to download it again next session.")
subprocess.run([sys.executable,"scripts/fetch_phoenix.py","--data-root",str(DATA_ROOT),"--archive",str(archive_path)],cwd=PROJECT_ROOT,check=True)
print("Project, PHOENIX data, and prior checkpoints are ready.")

In [ ]:
%pip install -q -r /kaggle/working/SignLanguage-Reproduce/requirements-notebook-cslr.txt

## Chuẩn bị manifest và cấu hình

In [ ]:
from pathlib import Path
import subprocess, sys

if DATASET in ("phoenix14t","phoenix14") and not (project_root/"assets/resnet18-f37072fd.pth").is_file():
    subprocess.run([sys.executable,"scripts/fetch_assets.py","--resnet","--output",
                    str(project_root/"assets")],cwd=project_root,check=True)
if PHASE in ("slt","gfslt") and not (project_root/"assets/mbart-large-cc25/config.json").is_file():
    subprocess.run([sys.executable,"scripts/fetch_assets.py","--mbart",
        "--mbart-revision","f417e5563320b2cc8aabe4329d986b238809067f",
        "--output",str(project_root/"assets")],cwd=project_root,check=True)

## Tải trọng số nếu cần

Mã sẽ tải ResNet/mBART vào vùng làm việc Kaggle. Cần bật Internet. Notebook có thể tải lại asset khi phiên Kaggle được tạo mới.

In [ ]:
import subprocess, sys
from pathlib import Path
manifest_dir=Path(MANIFEST_ROOT)
subprocess.run([sys.executable,"scripts/prepare_data.py","--dataset","phoenix14t",
    "--annotations",str(DATA_ROOT/"annotations/manual"),"--data-root",str(DATA_ROOT),
    "--output",str(manifest_dir),"--overwrite"],cwd=project_root,check=True)
for kind in ("swin_t","pvig_tiny"):
    subprocess.run([sys.executable,"scripts/fetch_backbones.py","--backbone",kind],cwd=project_root,check=True)
import torch, torchvision
if not torch.cuda.is_available(): raise RuntimeError("Enable GPU before running the experiment suite")
print("GPU:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("torch:",torch.__version__,"torchvision:",torchvision.__version__)

## Checkpoint từng thí nghiệm

Hàng đợi tự tạo `runs/phoenix14t_cslr_suite_checkpoint_v12_seed0/<experiment>/` trên cùng Drive. Mỗi checkpoint có checksum và ít nhất hai thế hệ để khôi phục nếu upload bị ngắt. Token chỉ nằm trong file tạm riêng của runtime, không lưu vào Output.

## Dữ liệu và checkpoint
Notebook tự tìm archive PHOENIX14T trong Drive; nếu chưa có, nó tải từ RWTH, cố gắng lưu archive vào Drive, rồi giải nén ở ổ tạm Kaggle. Kaggle giới hạn dung lượng output được lưu; dữ liệu lớn được giữ trong ổ tạm, còn checkpoint và kết quả được đồng bộ lên Drive. Cần Internet, dung lượng scratch đủ cho dữ liệu đã giải nén, và khoảng 39 GB trống trên Drive để lưu archive dùng chung.

In [ ]:
import os, subprocess, sys, tempfile, time
from pathlib import Path
env=os.environ.copy()
env["HF_HUB_OFFLINE"]="1"; env["TRANSFORMERS_OFFLINE"]="1"; env["PYTHONUNBUFFERED"]="1"
suite_output=project_root/"runs"/CAMPAIGN_NAME
args=[sys.executable,"scripts/run_suite.py","--data-root",str(DATA_ROOT),
      "--manifests",str(MANIFEST_ROOT),"--output",str(suite_output),
      "--minutes",str(max(3,SESSION_MINUTES-(time.monotonic()-NOTEBOOK_START)/60)),"--epochs",str(EPOCHS),"--workers",str(WORKERS)]
if ONLY_EXPERIMENTS: args += ["--only",ONLY_EXPERIMENTS]
credentials_file=None
try:
    if "drive_service" in globals():
        from kaggle_secrets import UserSecretsClient
        fd,name=tempfile.mkstemp(prefix="signlanguage-private-",suffix=".json",dir="/tmp")
        credentials_file=Path(name)
        with os.fdopen(fd,"w") as f:
            f.write(UserSecretsClient().get_secret("GOOGLE_DRIVE_OAUTH_JSON"))
        os.chmod(credentials_file,0o600)
        env["SIGN_DRIVE_CREDENTIALS"]=str(credentials_file)
        args += ["--drive-folder",DRIVE_WORKSPACE_FOLDER_ID]
    else:
        args += ["--persist-root",str(SHARED_DRIVE_ROOT/"runs"/CAMPAIGN_NAME)]
    process=subprocess.Popen(args,cwd=project_root,env=env)
    try:
        code=process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=120)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    if code: raise RuntimeError("Campaign stopped with an error; inspect output. Saved checkpoints remain available.")
    print("Session finished. Run All again to resume pending experiments; see suite_summary.csv on Drive.")
finally:
    if credentials_file: credentials_file.unlink(missing_ok=True)

## Tiếp tục

Chạy lại Run All: thí nghiệm hoàn tất được bỏ qua, thí nghiệm dở khôi phục checkpoint gần nhất. Mặc định không đổi batch/LR/epoch khi số GPU thay đổi. `suite_summary.csv` trên Drive tổng hợp số đo và độ lệch paper. Archive và checkpoint cần dung lượng Drive đủ; code dừng rõ nếu không lưu được checkpoint.

## Bản cập nhật không CI
Notebook vẫn dùng Run all và thư mục Drive chung; không cần thêm secret W&B. Khi code đổi, thay ZIP mới trong Drive trước phiên chạy mới. Không chạy Colab và Kaggle đồng thời vào cùng chiến dịch.
V12 tính lại activation theo từng khối khi backward và chỉ lưu đầu vào ranh giới khối trên CPU, nhằm giảm cả VRAM lẫn RAM so với v11. WORKERS=0 tránh đọc sẵn nhiều batch. Cách này có thể chậm hơn; không đổi nhãn, frame hoặc batch. Tên CAMPAIGN_NAME mặc định mới giữ kết quả/checkpoint các bản cũ; không dùng tên chiến dịch cũ với fingerprint code mới.
Máy Linux của cô có luồng riêng: service đang cài dùng `--single`, chỉ chạy thí nghiệm chính. Supervisor tự lấy bản sửa từ GitHub khi lượt chạy đã dừng và gửi lỗi lên W&B. Muốn đủ 68 lượt phải chọn `--suite`. Xem `docs/MAY_CO_VA_LAPTOP.md`. Cả ba nơi dùng cùng catalog PHOENIX14T–CSLR; chưa có số đo thật để khẳng định tái lập paper.